# Dashboard de Presión Arterial

Análisis interactivo de 59 pacientes: edad, altura, peso, presión sistólica y clasificación clínica de riesgo.

**Código de color** (semántico, por severidad clínica):

| Clasificación | Color |
|---|---|
| Normal | 🔵 Azul |
| Normal Alta | 🟢 Verde |
| Alta | 🟠 Ámbar |
| Muy Alta | 🔴 Rojo |
| Crisis Hipertensiva | 🔴 Rojo oscuro |

> Ejecuta las celdas en orden (Runtime → Run all en Colab, o Kernel → Restart & Run All en Jupyter).

## 1. Instalación de dependencias

Si ejecutas este notebook en **Jupyter local**, primero selecciona el kernel **"Python (jupyter-colab venv)"** (selector de kernel arriba a la derecha del editor, o Kernel → Change kernel) — ahí ya están instaladas las dependencias.

La celda siguiente instala automáticamente lo que falte, tanto en **Google Colab** como si por error corres con otro kernel/entorno.

In [ ]:
import sys
import subprocess
import importlib.util

IN_COLAB = "google.colab" in sys.modules

REQUERIDOS = ["pandas", "matplotlib", "plotly", "ipywidgets"]
faltantes = [pkg for pkg in REQUERIDOS if importlib.util.find_spec(pkg) is None]

print(f"Kernel activo: {sys.executable}")

if faltantes:
    print(f"Instalando dependencias faltantes: {', '.join(faltantes)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)
    importlib.invalidate_caches()
    print("Instalación completa.")
else:
    print("Dependencias OK.")

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

## 2. Carga de datos

En Colab, si no clonas el repo, sube manualmente `dataset.csv` con el selector de archivos.

In [ ]:
import sys
import subprocess

try:
    import pandas as pd
except ModuleNotFoundError:
    print(f"pandas no está instalado en este kernel ({sys.executable}). Instalando...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)
    import pandas as pd

from pathlib import Path

CSV_CANDIDATES = [
    Path("../dataset/dataset.csv"),
    Path("dataset.csv"),
]

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not any(p.exists() for p in CSV_CANDIDATES):
    from google.colab import files
    uploaded = files.upload()
    csv_path = Path(next(iter(uploaded)))
else:
    csv_path = next(p for p in CSV_CANDIDATES if p.exists())

df = pd.read_csv(csv_path, sep="\t")
df.head()

## 3. Paleta de color por clasificación

In [ ]:
ORDEN_CLASIFICACION = ["Normal", "Normal Alta", "Alta", "Muy Alta", "Crisis Hipertensiva"]

COLORES = {
    "Normal": "#2f6fed",              # azul - presión normal / baja
    "Normal Alta": "#2a9d6f",         # verde
    "Alta": "#d9971a",                # ámbar
    "Muy Alta": "#dd3b2a",            # rojo - hipertensión muy alta
    "Crisis Hipertensiva": "#a41d17", # rojo oscuro - crisis hipertensiva
}

GENERO_LABEL = {"M": "Mujer", "H": "Hombre"}

df["Clasificacion_Presion"] = pd.Categorical(
    df["Clasificacion_Presion"], categories=ORDEN_CLASIFICACION, ordered=True
)
df["Genero_Label"] = df["Genero"].map(GENERO_LABEL)

## 4. KPIs

In [ ]:
from IPython.display import HTML

total = len(df)
promedio = df["Presion_Sistolica"].mean()
altas = df["Clasificacion_Presion"].isin(["Muy Alta", "Crisis Hipertensiva"]).sum()
normales = (df["Clasificacion_Presion"] == "Normal").sum()

kpi_html = f"""
<div style="display:flex;gap:14px;flex-wrap:wrap;font-family:sans-serif;">
  <div style="flex:1;min-width:150px;border:1px solid #e1e4ea;border-radius:10px;padding:14px 16px;">
    <div style="font-size:.76rem;color:#5a6070;text-transform:uppercase;">Pacientes</div>
    <div style="font-size:1.6rem;font-weight:700;">{total}</div>
  </div>
  <div style="flex:1;min-width:150px;border:1px solid #e1e4ea;border-radius:10px;padding:14px 16px;">
    <div style="font-size:.76rem;color:#5a6070;text-transform:uppercase;">Presión promedio</div>
    <div style="font-size:1.6rem;font-weight:700;">{promedio:.1f} mmHg</div>
  </div>
  <div style="flex:1;min-width:150px;border:1px solid #e1e4ea;border-radius:10px;padding:14px 16px;">
    <div style="font-size:.76rem;color:#5a6070;text-transform:uppercase;">Muy Alta / Crisis</div>
    <div style="font-size:1.6rem;font-weight:700;color:{COLORES['Muy Alta']};">{altas} ({altas/total:.0%})</div>
  </div>
  <div style="flex:1;min-width:150px;border:1px solid #e1e4ea;border-radius:10px;padding:14px 16px;">
    <div style="font-size:.76rem;color:#5a6070;text-transform:uppercase;">Presión Normal</div>
    <div style="font-size:1.6rem;font-weight:700;color:{COLORES['Normal']};">{normales} ({normales/total:.0%})</div>
  </div>
</div>
"""
HTML(kpi_html)

## 5. Distribución por clasificación de presión

In [ ]:
import sys
import subprocess

try:
    import plotly.graph_objects as go
except ModuleNotFoundError:
    print(f"plotly no está instalado en este kernel ({sys.executable}). Instalando...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=True)
    import plotly.graph_objects as go

conteos = df["Clasificacion_Presion"].value_counts().reindex(ORDEN_CLASIFICACION).fillna(0)

fig_dist = go.Figure(
    go.Bar(
        x=ORDEN_CLASIFICACION,
        y=conteos.values,
        marker_color=[COLORES[c] for c in ORDEN_CLASIFICACION],
        text=conteos.values.astype(int),
        textposition="outside",
    )
)

fig_dist.update_layout(
    title="Distribución por clasificación de presión",
    xaxis_title=None,
    yaxis_title="Pacientes",
    template="plotly_white",
    height=380,
    showlegend=False,
)
fig_dist.show()

## 6. Edad vs. presión sistólica

In [ ]:
fig_scatter = go.Figure()
for c in ORDEN_CLASIFICACION:
    sub = df[df["Clasificacion_Presion"] == c]
    fig_scatter.add_trace(
        go.Scatter(
            x=sub["Edad"],
            y=sub["Presion_Sistolica"],
            mode="markers",
            name=c,
            marker=dict(color=COLORES[c], size=9, line=dict(width=1, color="white")),
            customdata=sub[["ID", "Genero_Label"]],
            hovertemplate="Paciente #%{customdata[0]} (%{customdata[1]})<br>Edad %{x} · Presión %{y} mmHg<extra>" + c + "</extra>",
        )
    )
fig_scatter.update_layout(
    title="Edad vs. presión sistólica",
    xaxis_title="Edad (años)",
    yaxis_title="Presión sistólica (mmHg)",
    template="plotly_white",
    height=420,
    legend_title="Clasificación",
)
fig_scatter.show()

## 7. Presión sistólica por paciente

In [ ]:
df_ordenado = df.sort_values("ID")

fig_barras = go.Figure(
    go.Bar(
        x=df_ordenado["ID"].astype(str),
        y=df_ordenado["Presion_Sistolica"],
        marker_color=df_ordenado["Clasificacion_Presion"].map(COLORES),
        customdata=df_ordenado[["Clasificacion_Presion", "Genero_Label", "Edad"]],
        hovertemplate="Paciente #%{x} · %{y} mmHg<br>%{customdata[0]} · %{customdata[1]}, %{customdata[2]} años<extra></extra>",
    )
)
fig_barras.update_layout(
    title="Presión sistólica por paciente",
    xaxis_title="ID de paciente",
    yaxis_title="mmHg",
    template="plotly_white",
    height=380,
    showlegend=False,
)
fig_barras.show()

## 8. Filtros interactivos (widgets)

In [ ]:
import sys
import subprocess

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    print(f"ipywidgets no está instalado en este kernel ({sys.executable}). Instalando...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"], check=True)
    import ipywidgets as widgets

from IPython.display import display, clear_output

clasificacion_w = widgets.Dropdown(
    options=["Todas"] + ORDEN_CLASIFICACION, value="Todas", description="Clasificación:"
)
genero_w = widgets.Dropdown(
    options=[("Todos", "Todos"), ("Mujer", "M"), ("Hombre", "H")], value="Todos", description="Género:"
)
salida = widgets.Output()

def actualizar(*_):
    with salida:
        clear_output(wait=True)
        datos = df.copy()
        if clasificacion_w.value != "Todas":
            datos = datos[datos["Clasificacion_Presion"] == clasificacion_w.value]
        if genero_w.value != "Todos":
            datos = datos[datos["Genero"] == genero_w.value]

        display(HTML(f"<b>{len(datos)}</b> paciente(s) · presión promedio "
                      f"<b>{datos['Presion_Sistolica'].mean():.1f} mmHg</b>" if len(datos) else "Sin resultados"))

        tabla = datos.sort_values("ID")[
            ["ID", "Edad", "Genero_Label", "Altura", "Peso", "Presion_Sistolica", "Clasificacion_Presion"]
        ].rename(columns={
            "Genero_Label": "Género", "Altura": "Altura (m)", "Peso": "Peso (kg)",
            "Presion_Sistolica": "Presión (mmHg)", "Clasificacion_Presion": "Clasificación",
        })

        def resaltar(fila):
            color = COLORES.get(fila["Clasificación"], "")
            return [f"background-color:{color}22; color:{color}" if col == "Clasificación" else "" for col in fila.index]

        display(tabla.style.apply(resaltar, axis=1).hide(axis="index"))

clasificacion_w.observe(actualizar, names="value")
genero_w.observe(actualizar, names="value")

display(widgets.HBox([clasificacion_w, genero_w]), salida)
actualizar()